# Feature groups and leakage safe weather anomalies

This notebook performs Task 1 and Task 2 for the Atlanta and Birmingham LSTM pipeline.

It does **not** train an LSTM.

The notebook assumes this exact project structure:

```text
LSTM_DATA_PREP_FEATURES/
│
├── feature_groups_and_weather_anomalies.ipynb
│
├── data/
│   ├── atlanta_master.csv
│   └── birmingham_master.csv
│
└── prepared_lstm/
    ├── atlanta/
    │   ├── features_scaled.npy
    │   ├── target_scaled.npy
    │   ├── timestamps_ns.npy
    │   ├── target_timestamps_ns.npy
    │   ├── split_codes.npy
    │   ├── row_index.csv
    │   ├── transformers.joblib
    │   └── metadata.json
    │
    └── birmingham/
        └── same preparation artifacts
```

The existing prepared LSTM artifacts are treated as the source of the frozen 44 feature matrix, chronological split, and row alignment. Weather climatology is calculated from **2016–2021 only** and then applied unchanged to 2022 and 2023.

## 1. Setup and exact paths

In [2]:
from __future__ import annotations

import json
from dataclasses import dataclass, asdict
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)

# The notebook is intended to be run from LSTM_DATA_PREP_FEATURES/.
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / 'data'
PREPARED_DIR = PROJECT_DIR / 'prepared_lstm'

FILES = {
    'Atlanta': DATA_DIR / 'atlanta_master.csv',
    'Birmingham': DATA_DIR / 'birmingham_master.csv',
}

CITY_DIRS = {
    city: PREPARED_DIR / city.lower()
    for city in FILES
}

VALIDATION_START = pd.Timestamp('2022-01-01 00:00:00')
TEST_START = pd.Timestamp('2023-01-01 00:00:00')
TRAIN_END = VALIDATION_START - pd.Timedelta(hours=1)

for path in FILES.values():
    if not path.exists():
        raise FileNotFoundError(f'Missing required master file: {path}')

for city_dir in CITY_DIRS.values():
    if not city_dir.exists():
        raise FileNotFoundError(f'Missing prepared city directory: {city_dir}')

print('Project:', PROJECT_DIR.resolve())
print('Data:', DATA_DIR.resolve())
print('Prepared artifacts:', PREPARED_DIR.resolve())

Project: D:\Downloads\BTP\model_training\lstm_training_2\LSTM_GPU_TRAINING\LSTM_DATA_PREP
Data: D:\Downloads\BTP\model_training\lstm_training_2\LSTM_GPU_TRAINING\LSTM_DATA_PREP\data
Prepared artifacts: D:\Downloads\BTP\model_training\lstm_training_2\LSTM_GPU_TRAINING\LSTM_DATA_PREP\prepared_lstm


## 2. Load the frozen preparation artifacts

The existing 44 feature preparation is not rebuilt here. This notebook uses the already prepared arrays and metadata so that Task 1 and Task 2 do not silently alter the original preprocessing pipeline.

In [3]:
raw = {}
metadata = {}
row_index = {}
base_features = {}

for city, csv_path in FILES.items():
    city_dir = CITY_DIRS[city]

    raw[city] = pd.read_csv(csv_path, parse_dates=['timestamp'])
    raw[city] = raw[city].sort_values('timestamp').reset_index(drop=True)

    metadata[city] = json.loads(
        (city_dir / 'metadata.json').read_text(encoding='utf-8')
    )

    row_index[city] = pd.read_csv(
        city_dir / 'row_index.csv',
        parse_dates=['timestamp', 'target_timestamp']
    )

    base_features[city] = list(metadata[city]['feature_names'])

    print(f'{city}:')
    print('  raw rows:', len(raw[city]))
    print('  prepared rows:', len(row_index[city]))
    print('  prepared features:', len(base_features[city]))
    print('  lookback:', metadata[city]['lookback_hours'])
    print('  horizon:', metadata[city]['horizon_hours'])

assert base_features['Atlanta'] == base_features['Birmingham'], 'Atlanta and Birmingham feature schemas differ.'
BASE_FEATURES = base_features['Atlanta']
print('\nShared frozen feature count:', len(BASE_FEATURES))

Atlanta:
  raw rows: 70080
  prepared rows: 69911
  prepared features: 45
  lookback: 168
  horizon: 1
Birmingham:
  raw rows: 70080
  prepared rows: 69911
  prepared features: 45
  lookback: 168
  horizon: 1

Shared frozen feature count: 45


## 3. Task 1 — explicit feature groups

The experiment hierarchy is:

```text
Baseline 0 → historical hour × weekday schedule

Model A → weather + engineered weather + short term weather + calendar

Model B → Model A + historical load

Model C → Model B + long term time index

Model D → Model C + leakage safe weather anomalies
```

The same feature definitions are used for both cities.

In [4]:
WEATHER_FEATURES = [
    'extraterrestrial_horizontal_radiation_wh_m2',
    'total_sky_cover_tenths',
    'opaque_sky_cover_tenths',
    'temperature_c',
    'relative_humidity_percent',
    'station_pressure_kpa',
    'wind_speed_ms',
    'horizontal_visibility_km',
    'ceiling_height_m',
    'precipitable_water_mm',
    'broadband_aerosol_optical_depth',
    'snow_depth_cm',
    'days_since_last_snowfall',
    'hourly_precipitation_cm',
]

WEATHER_ENGINEERED_FEATURES = [
    'temperature_squared',
    'cooling_degree_24',
    'heating_degree_18',
    'temp_humidity_interaction',
    'wind_dir_sin',
    'wind_dir_cos',
]

WEATHER_HISTORY_FEATURES = [
    'temperature_c_lag_1h',
    'temperature_c_roll_mean_24h',
    'wind_speed_ms_lag_1h',
    'wind_speed_ms_roll_mean_24h',
]

CALENDAR_FEATURES = [
    'hour_sin',
    'hour_cos',
    'dow_sin',
    'dow_cos',
    'doy_sin',
    'doy_cos',
    'is_weekend',
]

LOAD_HISTORY_FEATURES = [
    'load_lag_1h',
    'load_lag_2h',
    'load_lag_3h',
    'load_lag_24h',
    'load_lag_48h',
    'load_lag_168h',
    'load_roll_mean_3h',
    'load_roll_std_3h',
    'load_roll_mean_24h',
    'load_roll_std_24h',
    'load_roll_mean_168h',
    'load_roll_std_168h',
]

TIME_FEATURES = ['time_index']

ANOMALY_BASE_FEATURES = [
    'temperature_c',
    'relative_humidity_percent',
    'station_pressure_kpa',
    'wind_speed_ms',
    'horizontal_visibility_km',
    'total_sky_cover_tenths',
    'precipitable_water_mm',
]

WEATHER_ANOMALY_FEATURES = [
    f'{feature}_anomaly'
    for feature in ANOMALY_BASE_FEATURES
]

MODEL_A_FEATURES = (
    WEATHER_FEATURES
    + WEATHER_ENGINEERED_FEATURES
    + WEATHER_HISTORY_FEATURES
    + CALENDAR_FEATURES
)

MODEL_B_FEATURES = MODEL_A_FEATURES + LOAD_HISTORY_FEATURES
MODEL_C_FEATURES = MODEL_B_FEATURES + TIME_FEATURES
MODEL_D_FEATURES = MODEL_C_FEATURES + WEATHER_ANOMALY_FEATURES

FEATURE_GROUPS = {
    'Baseline_0': {
        'type': 'non_lstm_historical_schedule',
        'features': ['hour_of_day', 'day_of_week'],
        'description': 'Historical mean load by hour of day and day of week, fitted on 2016–2021 only.'
    },
    'Model_A': {
        'features': MODEL_A_FEATURES,
        'description': 'Weather, engineered weather, short term weather history, and calendar features.'
    },
    'Model_B': {
        'features': MODEL_B_FEATURES,
        'description': 'Model A plus historical load lags and rolling load statistics.'
    },
    'Model_C': {
        'features': MODEL_C_FEATURES,
        'description': 'Model B plus the long term chronological time index.'
    },
    'Model_D': {
        'features': MODEL_D_FEATURES,
        'description': 'Model C plus leakage safe weather anomalies calculated from 2016–2021 climatology.'
    }
}

for name, group in FEATURE_GROUPS.items():
    print(f'{name}: {len(group["features"])} features')

assert len(MODEL_A_FEATURES) == 31
assert len(MODEL_B_FEATURES) == 43
assert len(MODEL_C_FEATURES) == 44
assert len(MODEL_D_FEATURES) == 51

Baseline_0: 2 features
Model_A: 31 features
Model_B: 43 features
Model_C: 44 features
Model_D: 51 features


In [5]:
for name, features in [
    ('Model_A', MODEL_A_FEATURES),
    ('Model_B', MODEL_B_FEATURES),
    ('Model_C', MODEL_C_FEATURES),
]:
    missing = [feature for feature in features if feature not in BASE_FEATURES]
    if missing:
        raise ValueError(f'{name} contains features absent from the frozen 44-feature schema: {missing}')

print('All Model A/B/C features are present in the frozen 44-feature preparation.')

All Model A/B/C features are present in the frozen 44-feature preparation.


## 4. Task 2 — leakage safe weather climatology

Weather anomalies are defined as:

`observed weather − training climatological normal`

The climatological normal is grouped by **month, calendar day, and hour** rather than raw day of year. This avoids the leap-day indexing problem in the supplied data.

Only observations from **2016–2021** are allowed to contribute to the climatology. Validation and test observations are never used to calculate the normal or its standard deviation.

For the primary Model D experiment, absolute anomalies are used. Standardized anomalies are intentionally not added to Model D in this notebook.

In [6]:
def add_calendar_keys(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['month'] = out['timestamp'].dt.month.astype('int8')
    out['day'] = out['timestamp'].dt.day.astype('int8')
    out['hour'] = out['timestamp'].dt.hour.astype('int8')
    return out


def fit_weather_climatology(df: pd.DataFrame) -> dict:
    work = add_calendar_keys(df)
    train = work[work['timestamp'] < VALIDATION_START].copy()

    if train.empty:
        raise ValueError('No 2016–2021 observations available for climatology fitting.')

    grouped = train.groupby(
        ['month', 'day', 'hour'],
        observed=True
    )

    mean_table = grouped[ANOMALY_BASE_FEATURES].mean()
    std_table = grouped[ANOMALY_BASE_FEATURES].std(ddof=0)
    count_table = grouped[ANOMALY_BASE_FEATURES].count()

    # A zero standard deviation is valid. It only matters if standardized anomalies are added later.
    return {
        'mean': mean_table,
        'std': std_table,
        'count': count_table,
        'group_keys': ['month', 'day', 'hour'],
        'fit_start': str(train['timestamp'].min()),
        'fit_end': str(train['timestamp'].max()),
        'source_period': '2016-2021',
        'variables': ANOMALY_BASE_FEATURES,
    }


climatology = {
    city: fit_weather_climatology(df)
    for city, df in raw.items()
}

for city, clim in climatology.items():
    print(
        f'{city}: climatology fitted on '
        f'{clim["fit_start"]} → {clim["fit_end"]}; '
        f'{len(clim["mean"])} month/day/hour groups'
    )

Atlanta: climatology fitted on 2016-01-01 00:00:00 → 2021-12-31 23:00:00; 8760 month/day/hour groups
Birmingham: climatology fitted on 2016-01-01 00:00:00 → 2021-12-31 23:00:00; 8760 month/day/hour groups


## 5. Generate absolute weather anomalies

For each city and timestamp:

`feature_anomaly = observed_feature − training_climatological_mean`

If a month/day/hour climatology is unavailable because of missing source observations, the anomaly remains missing and is handled later using the training-only imputer.

In [7]:
def apply_weather_climatology(df: pd.DataFrame, clim: dict) -> pd.DataFrame:
    work = add_calendar_keys(df)
    keys = ['month', 'day', 'hour']

    means = clim['mean'].reset_index()
    means = means.rename(columns={
        feature: f'{feature}_climatology'
        for feature in ANOMALY_BASE_FEATURES
    })

    out = work.merge(means, on=keys, how='left', validate='many_to_one')

    for feature in ANOMALY_BASE_FEATURES:
        out[f'{feature}_anomaly'] = (
            out[feature] - out[f'{feature}_climatology']
        )

    anomaly_cols = WEATHER_ANOMALY_FEATURES
    return out[['timestamp', 'target_timestamp'] if 'target_timestamp' in out.columns else ['timestamp'] + anomaly_cols]


anomalies = {}

for city in raw:
    anomalies[city] = apply_weather_climatology(
        raw[city],
        climatology[city]
    )

    print(f'\n{city}')
    display(anomalies[city][['timestamp'] + WEATHER_ANOMALY_FEATURES].head())
    display(anomalies[city][WEATHER_ANOMALY_FEATURES].describe().T)


Atlanta


,timestamp,temperature_c_anomaly,relative_humidity_percent_anomaly,station_pressure_kpa_anomaly,wind_speed_ms_anomaly,horizontal_visibility_km_anomaly,total_sky_cover_tenths_anomaly,precipitable_water_mm_anomaly
0,2016-01-01 00:00:00,-1.866667,13.333333,0.200000,-2.216667,-13.383333,2.000000,3.000000
1,2016-01-01 01:00:00,-1.900000,14.166667,0.216667,-1.183333,-12.850000,2.833333,3.500000
2,2016-01-01 02:00:00,-1.500000,13.333333,0.316667,-0.566667,-12.866667,3.333333,5.333333
3,2016-01-01 03:00:00,-1.150000,10.500000,0.166667,-1.133333,-13.933333,3.333333,6.000000
4,2016-01-01 04:00:00,-0.783333,13.833333,0.200000,-3.750000,-11.950000,2.500000,6.833333


,count,mean,std,min,25%,50%,75%,max
temperature_c_anomaly,70080.0,0.235643,4.428923,-22.733333,-2.416667,0.316667,2.900000,19.016667
relative_humidity_percent_anomaly,70080.0,0.399962,16.402264,-62.833333,-11.000000,-0.333333,11.000000,67.333333
station_pressure_kpa_anomaly,70080.0,0.025751,0.471732,-2.983333,-0.250000,0.016667,0.316667,2.300000
wind_speed_ms_anomaly,70080.0,-0.057186,1.963369,-7.966667,-1.283333,-0.166667,1.050000,22.983333
horizontal_visibility_km_anomaly,70080.0,0.041796,8.868440,-37.683333,-5.616667,-0.283333,5.383333,37.266667
total_sky_cover_tenths_anomaly,70080.0,0.036368,4.000425,-10.000000,-3.333333,0.166667,3.333333,10.000000
precipitable_water_mm_anomaly,70080.0,-0.111910,7.371104,-23.333333,-5.500000,-0.666667,5.000000,44.000000



Birmingham


,timestamp,temperature_c_anomaly,relative_humidity_percent_anomaly,station_pressure_kpa_anomaly,wind_speed_ms_anomaly,horizontal_visibility_km_anomaly,total_sky_cover_tenths_anomaly,precipitable_water_mm_anomaly
0,2016-01-01 00:00:00,0.433333,2.166667,0.150000,-0.950000,-0.916667,3.333333,-0.833333
1,2016-01-01 01:00:00,0.200000,3.833333,0.133333,-1.916667,0.416667,3.333333,-1.166667
2,2016-01-01 02:00:00,1.200000,-3.833333,0.150000,0.550000,0.416667,3.500000,-0.833333
3,2016-01-01 03:00:00,1.466667,-4.833333,0.166667,-2.783333,0.416667,4.166667,-1.000000
4,2016-01-01 04:00:00,1.516667,-4.666667,0.166667,-2.133333,0.416667,5.000000,-1.000000


,count,mean,std,min,25%,50%,75%,max
temperature_c_anomaly,70080.0,0.144905,4.813187,-24.016667,-2.733333,0.250000,3.050000,20.383333
relative_humidity_percent_anomaly,70080.0,0.266657,15.651999,-54.500000,-10.000000,0.333333,10.000000,70.000000
station_pressure_kpa_anomaly,70080.0,0.018457,0.477428,-3.050000,-0.250000,0.016667,0.300000,2.266667
wind_speed_ms_anomaly,70080.0,0.028138,1.767100,-6.100000,-1.166667,0.016667,1.116667,15.916667
horizontal_visibility_km_anomaly,70080.0,0.607889,4.150321,-14.833333,-1.666667,0.266667,2.700000,36.966667
total_sky_cover_tenths_anomaly,70080.0,0.178981,3.866243,-10.000000,-3.166667,0.333333,3.333333,10.000000
precipitable_water_mm_anomaly,70080.0,0.150157,6.048303,-18.666667,-4.000000,-0.166667,4.166667,24.166667


## 6. Leakage checks

These checks verify that the anomaly climatology is based only on the training interval and that the generated anomaly values align with the original hourly timestamps.

In [8]:
for city in raw:
    clim = climatology[city]
    assert pd.Timestamp(clim['fit_end']) < VALIDATION_START
    assert clim['source_period'] == '2016-2021'

    # Every climatology key must come from a training observation.
    training = add_calendar_keys(
        raw[city][raw[city]['timestamp'] < VALIDATION_START]
    )
    training_keys = set(map(tuple, training[['month', 'day', 'hour']].drop_duplicates().to_numpy()))
    climatology_keys = set(map(tuple, clim['mean'].reset_index()[['month', 'day', 'hour']].to_numpy()))
    assert climatology_keys.issubset(training_keys)

    # Verify timestamp alignment with the raw master file.
    assert anomalies[city]['timestamp'].equals(raw[city]['timestamp'])

    print(f'{city}: leakage and timestamp alignment checks passed.')

Atlanta: leakage and timestamp alignment checks passed.
Birmingham: leakage and timestamp alignment checks passed.


## 7. Align anomalies to the frozen prepared rows

The raw master files contain 70,080 hourly observations, while the prepared LSTM row index contains the usable supervised rows. We therefore align anomaly rows by timestamp rather than by raw row number.

In [9]:
aligned_anomalies = {}

for city in raw:
    idx = row_index[city][['timestamp', 'target_timestamp', 'split']].copy()
    anom = anomalies[city][['timestamp'] + WEATHER_ANOMALY_FEATURES].copy()

    merged = idx.merge(
        anom,
        on='timestamp',
        how='left',
        validate='one_to_one'
    )

    if len(merged) != len(idx):
        raise ValueError(f'{city}: anomaly alignment changed the prepared row count.')

    if not merged['target_timestamp'].equals(idx['target_timestamp']):
        raise ValueError(f'{city}: target timestamp alignment changed during anomaly merge.')

    aligned_anomalies[city] = merged

    print(
        f'{city}: aligned anomaly rows={len(merged)}, '
        f'columns={len(WEATHER_ANOMALY_FEATURES)}'
    )

Atlanta: aligned anomaly rows=69911, columns=7
Birmingham: aligned anomaly rows=69911, columns=7


## 8. Build the Model D feature artifact

The existing 44-feature array is already scaled using the original train-only preprocessing. The seven anomaly features require their own train-only imputation and scaling.

Model D is therefore saved as:

`existing 44 scaled features + 7 scaled anomaly features = 51 features`

No validation or test anomaly values are used to fit the anomaly imputer or scaler.

In [10]:
model_d_artifacts = {}

for city in raw:
    city_dir = CITY_DIRS[city]

    base_X = np.load(
        city_dir / 'features_scaled.npy',
        mmap_mode='r'
    )

    split_codes = np.load(
        city_dir / 'split_codes.npy',
        mmap_mode='r'
    )

    # The saved base matrix has 45 features.
    # Select the intended 44 Model C features by name.
    base_feature_names = metadata[city]['feature_names']

    missing_model_c = [
        f for f in MODEL_C_FEATURES
        if f not in base_feature_names
    ]

    if missing_model_c:
        raise ValueError(
            f'{city}: Model C features missing from base feature schema: '
            f'{missing_model_c}'
        )

    model_c_indices = [
        base_feature_names.index(f)
        for f in MODEL_C_FEATURES
    ]

    model_c_X = base_X[:, model_c_indices]

    assert model_c_X.shape[1] == 44

    print(
        f'{city}: base matrix = {base_X.shape[1]} features, '
        f'Model C matrix = {model_c_X.shape[1]} features'
    )

    train_mask = split_codes == 0

    anomaly_imputer = SimpleImputer(strategy='median')
    anomaly_scaler = StandardScaler()

    anomaly_raw = aligned_anomalies[city][WEATHER_ANOMALY_FEATURES]

    anomaly_train_imputed = anomaly_imputer.fit_transform(
        anomaly_raw.loc[train_mask]
    )

    anomaly_scaler.fit(anomaly_train_imputed)

    anomaly_scaled = anomaly_scaler.transform(
        anomaly_imputer.transform(anomaly_raw)
    ).astype(np.float32)

    assert anomaly_scaled.shape[1] == 7

    # IMPORTANT:
    # Model D uses the 44 Model C features, NOT the original 45-feature matrix.
    model_d_X = np.concatenate(
        [
            np.asarray(model_c_X, dtype=np.float32),
            anomaly_scaled
        ],
        axis=1
    ).astype(np.float32, copy=False)

    assert model_d_X.shape[1] == 51

    # Make sure the output directory exists.
    out_dir = Path(city_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    np.save(
        out_dir / 'weather_anomalies_scaled.npy',
        anomaly_scaled
    )

    np.save(
        out_dir / 'features_model_d_scaled.npy',
        model_d_X
    )

    joblib.dump(
        {
            'imputer': anomaly_imputer,
            'feature_scaler': anomaly_scaler,
            'feature_names': WEATHER_ANOMALY_FEATURES,
            'fit_split': '2016-2021',
        },
        out_dir / 'weather_anomaly_transformers.joblib'
    )

    aligned_anomalies[city].to_csv(
        out_dir / 'weather_anomalies.csv',
        index=False
    )

    model_d_artifacts[city] = {
        'base_shape': list(base_X.shape),
        'model_c_shape': list(model_c_X.shape),
        'anomaly_shape': list(anomaly_scaled.shape),
        'model_d_shape': list(model_d_X.shape),
    }

    print(
        f'{city}: base={base_X.shape}, '
        f'Model C={model_c_X.shape}, '
        f'anomalies={anomaly_scaled.shape}, '
        f'Model D={model_d_X.shape}'
    )

Atlanta: base matrix = 45 features, Model C matrix = 44 features


Atlanta: base=(69911, 45), Model C=(69911, 44), anomalies=(69911, 7), Model D=(69911, 51)
Birmingham: base matrix = 45 features, Model C matrix = 44 features
Birmingham: base=(69911, 45), Model C=(69911, 44), anomalies=(69911, 7), Model D=(69911, 51)


## 9. Save feature group and climatology artifacts

Each city receives its own `feature_groups.json` and `weather_climatology.joblib`. The feature group definition is identical across cities, while the climatological values are city specific.

In [11]:
for city in raw:
    city_dir = CITY_DIRS[city]

    feature_group_artifact = {
        'city': city,
        'baseline_0': FEATURE_GROUPS['Baseline_0'],
        'model_a': FEATURE_GROUPS['Model_A'],
        'model_b': FEATURE_GROUPS['Model_B'],
        'model_c': FEATURE_GROUPS['Model_C'],
        'model_d': FEATURE_GROUPS['Model_D'],
        'base_feature_count': len(BASE_FEATURES),
        'model_a_feature_count': len(MODEL_A_FEATURES),
        'model_b_feature_count': len(MODEL_B_FEATURES),
        'model_c_feature_count': len(MODEL_C_FEATURES),
        'model_d_feature_count': len(MODEL_D_FEATURES),
        'lookback_hours': metadata[city]['lookback_hours'],
        'horizon_hours': metadata[city]['horizon_hours'],
        'validation_start': str(VALIDATION_START),
        'test_start': str(TEST_START),
    }

    (city_dir / 'feature_groups.json').write_text(
        json.dumps(feature_group_artifact, indent=2),
        encoding='utf-8'
    )

    joblib.dump(
        climatology[city],
        city_dir / 'weather_climatology.joblib'
    )

    print(f'{city}: feature_groups.json and weather_climatology.joblib saved.')

Atlanta: feature_groups.json and weather_climatology.joblib saved.
Birmingham: feature_groups.json and weather_climatology.joblib saved.


## 10. Final validation

This cell verifies the final experiment contract without training a model.

In [12]:
for city in raw:
    city_dir = CITY_DIRS[city]

    feature_groups_path = city_dir / 'feature_groups.json'
    climatology_path = city_dir / 'weather_climatology.joblib'
    anomaly_path = city_dir / 'weather_anomalies_scaled.npy'
    model_d_path = city_dir / 'features_model_d_scaled.npy'
    transformer_path = city_dir / 'weather_anomaly_transformers.joblib'

    for path in [feature_groups_path, climatology_path, anomaly_path, model_d_path, transformer_path]:
        assert path.exists(), f'{city}: missing artifact {path.name}'

    model_d = np.load(model_d_path, mmap_mode='r')
    assert model_d.shape[1] == 51
    assert model_d.shape[0] == len(row_index[city])

    saved_groups = json.loads(feature_groups_path.read_text(encoding='utf-8'))
    assert saved_groups['model_c_feature_count'] == 44
    assert saved_groups['model_d_feature_count'] == 51

    saved_clim = joblib.load(climatology_path)
    assert saved_clim['source_period'] == '2016-2021'
    assert pd.Timestamp(saved_clim['fit_end']) < VALIDATION_START

    print(f'{city}: FINAL CHECK PASSED')
    print('  Model A:', saved_groups['model_a_feature_count'], 'features')
    print('  Model B:', saved_groups['model_b_feature_count'], 'features')
    print('  Model C:', saved_groups['model_c_feature_count'], 'features')
    print('  Model D:', saved_groups['model_d_feature_count'], 'features')
    print('  Model D matrix:', model_d.shape)

Atlanta: FINAL CHECK PASSED
  Model A: 31 features
  Model B: 43 features
  Model C: 44 features
  Model D: 51 features
  Model D matrix: (69911, 51)
Birmingham: FINAL CHECK PASSED
  Model A: 31 features
  Model B: 43 features
  Model C: 44 features
  Model D: 51 features
  Model D matrix: (69911, 51)


In [13]:
for city in raw:
    city_dir = CITY_DIRS[city]

    model_d_path = city_dir / 'features_model_d_scaled.npy'
    anomaly_path = city_dir / 'weather_anomalies_scaled.npy'

    model_d = np.load(model_d_path, mmap_mode='r')
    anomaly = np.load(anomaly_path, mmap_mode='r')

    print(f'\n{city}')
    print('Base features:', np.load(
        city_dir / 'features_scaled.npy',
        mmap_mode='r'
    ).shape)

    print('Anomaly features:', anomaly.shape)
    print('Model D:', model_d.shape)
    print('Expected Model D:', 51)


Atlanta
Base features: (69911, 45)
Anomaly features: (69911, 7)
Model D: (69911, 51)
Expected Model D: 51

Birmingham
Base features: (69911, 45)
Anomaly features: (69911, 7)
Model D: (69911, 51)
Expected Model D: 51


In [14]:
print("WEATHER_ANOMALY_FEATURES:")
for i, feature in enumerate(WEATHER_ANOMALY_FEATURES, 1):
    print(i, feature)

print("\nCount:", len(WEATHER_ANOMALY_FEATURES))

WEATHER_ANOMALY_FEATURES:
1 temperature_c_anomaly
2 relative_humidity_percent_anomaly
3 station_pressure_kpa_anomaly
4 wind_speed_ms_anomaly
5 horizontal_visibility_km_anomaly
6 total_sky_cover_tenths_anomaly
7 precipitable_water_mm_anomaly

Count: 7


## 11. Resulting artifacts

After this notebook finishes, each city directory contains the original preparation artifacts plus:

```text
feature_groups.json
weather_climatology.joblib
weather_anomalies.csv
weather_anomalies_scaled.npy
weather_anomaly_transformers.joblib
features_model_d_scaled.npy
```

The important training handoff is:

```text
Model A → select its 31 columns from the frozen prepared feature matrix
Model B → select its 43 columns from the frozen prepared feature matrix
Model C → use the full 44-column frozen prepared feature matrix
Model D → use features_model_d_scaled.npy with 51 columns
```

No LSTM model, optimizer, training loop, or GPU configuration is created by this notebook.